# News Search Engine — Demo

An end-to-end walkthrough of the information-retrieval pipeline: load the
corpus, build the inverted index, search with BM25, expand queries, and
evaluate. See [`src/news_search`](../src/news_search) for the implementation.

In [1]:
import sys
sys.path.insert(0, '../src')

from news_search import load_corpus, build_index, SearchEngine
from news_search import evaluate as ev

## 1. Load the corpus

Uses the committed sample by default. Swap the path for
`../data/News_Category_Dataset_v3.json` to run on all ~210k articles.

In [2]:
docs = load_corpus('../data/sample_news.jsonl')
print(f'Loaded {len(docs):,} documents')
docs[0]

Loaded 716 documents


Document(id=0, text='NHL Names Kid Rock As All-Star Game Act; Twitter Says What The Puck? "Why don\'t any women, minorities, or people under 40 watch our sport?"', headline='NHL Names Kid Rock As All-Star Game Act; Twitter Says What The Puck?', short_description='"Why don\'t any women, minorities, or people under 40 watch our sport?"', category='ENTERTAINMENT', date='2018-01-17', link='https://www.huffingtonpost.com/entry/nhl-kid-rock-all-star-game_us_5a5f7fefe4b096ecfca9bf21', authors='Ron Dicker')

## 2. Build the inverted index

Document lengths, IDF and average document length are precomputed once here,
so queries never re-tokenise documents.

In [3]:
index = build_index(docs)
print(f'{index.num_docs:,} docs | {index.vocabulary_size:,} terms | avg len {index.avg_doc_len:.1f}')
engine = SearchEngine(index)

Built index: 716 docs, 4,285 terms in 0.1s
716 docs | 4,285 terms | avg len 18.1


## 3. Search (BM25)

BM25 is the default ranker. Try `method='tfidf'` to see the difference —
plain TF-IDF tends to surface very short headlines.

In [4]:
res = engine.search('covid vaccine health', method='bm25', top_k=5)
print(f'{res.total_hits} hits in {res.elapsed_ms:.2f} ms')
for r in res.results:
    print(f"  #{r['rank']} [{r['category']}] {r['headline'][:60]}  (score {r['score']})")

5 hits in 0.26 ms
  #1 [WELLNESS] How to Resist Temptation and Actually Stick to Your Health G  (score 5.94973)
  #2 [STYLE & BEAUTY] Where To Get Meghan Markle's Black Scalloped Face Mask  (score 5.42303)
  #3 [POLITICS] Trump Is Trying To Thread A ‘Fine Needle’ On Health Care, Sa  (score 5.31346)
  #4 [SCIENCE] Scientists' New 'Human Placenta Project' Aims to Improve Hea  (score 5.1309)
  #5 [HEALTHY LIVING] Healthy Living Comics: Today, An All or Nothing Approach to   (score 5.12799)


## 4. Query expansion

Compare the expansion terms each method pulls in. (`bert` / `prf+bert`
require the optional dependencies in `requirements-bert.txt`.)

In [5]:
for method in ['bm25', 'prf', 'wordnet']:
    r = engine.search('election president', method=method, top_k=5)
    print(f"{method:8} | expansion: {r.expansion_terms}")

bm25     | expansion: []
prf      | expansion: ['trump', 'deepen', 'nigeria', 'divis', 'paulson']
wordnet  | expansion: ['chosen']


## 5. Evaluation

Precision@K / Recall / F1 against category-based pseudo-relevance judgments.
Precision@K is the meaningful metric (see the README for why). BM25 should
clearly beat plain TF-IDF.

In [6]:
queries = ['covid health', 'election president', 'movie film',
           'stock market', 'travel food', 'game sport']
for s in ev.evaluate(engine, queries, methods=['bm25', 'tfidf', 'prf', 'wordnet'], top_k=10):
    print(f'{s.method:8} P@10={s.precision:.3f}  R={s.recall:.4f}  F1={s.f1:.4f}  {s.avg_ms:.2f}ms')

bm25     P@10=0.283  R=0.0492  F1=0.0803  0.04ms
tfidf    P@10=0.300  R=0.0519  F1=0.0850  0.03ms
prf      P@10=0.300  R=0.0613  F1=0.0961  0.14ms
wordnet  P@10=0.250  R=0.0464  F1=0.0751  0.44ms
